<a href="https://colab.research.google.com/github/erikh-1126/04-Project-Group/blob/main/04_manual_mlps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression.

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:
   
Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task  
  - `MultiClassMLP` for the multi-class classification task  
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept).
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already.
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

---
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


In [11]:
NAME(s) = "Erik H., Derek W., Henry Y."

SyntaxError: cannot assign to function call here. Maybe you meant '==' instead of '='? (147228175.py, line 1)

## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

- What cost function should be used for each of the below task.
- What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.
- Why it is important to standardize the data before training each model.
- How you could detect overfitting when training your models.

USE AS MANY MARKDOWN CELLS AS NEEDED

## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [2]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

In [3]:
# USE AS MANY CELLS AS NEEDED

In [4]:
import torch


def sigmoid(z):
    return 1 / (1 + torch.exp(-z))


def relu(z):
    return torch.maximum(z, torch.tensor(0.0))


class BinaryMLP:

    def __init__(
        self,
        input_dim,
        hidden_dim,
        hidden_dim2=None,
        learning_rate=0.1,
        activation="relu"
    ):
        # Input layer -> first hidden layer
        self.W_1 = torch.randn(input_dim, hidden_dim) * 0.1
        self.b_1 = torch.zeros(hidden_dim)

        # Optional second hidden layer
        self.hidden_dim2 = hidden_dim2
        self.activation = activation

        if hidden_dim2 is not None:
            self.W_2 = torch.randn(hidden_dim, hidden_dim2) * 0.1
            self.b_2 = torch.zeros(hidden_dim2)
            output_input_dim = hidden_dim2
        else:
            output_input_dim = hidden_dim

        # Final hidden layer -> output layer
        self.W_out = torch.randn(output_input_dim, 1) * 0.1
        self.b_out = torch.zeros(1)

        self.learning_rate = learning_rate

    def forward(self, X):
        self.h_1 = X @ self.W_1 + self.b_1

        if self.activation == "relu":
            self.a_1 = torch.relu(self.h_1)
        else:
            self.a_1 = torch.tanh(self.h_1)

        if self.hidden_dim2 is not None:
            self.h_2 = self.a_1 @ self.W_2 + self.b_2

            if self.activation == "relu":
                self.a_2 = torch.relu(self.h_2)
            else:
                self.a_2 = torch.tanh(self.h_2)

            hidden_output = self.a_2
        else:
            hidden_output = self.a_1

        output = hidden_output @ self.W_out + self.b_out
        return sigmoid(output)

    def cost(self, y_pred, y_true):
        # Binary cross-entropy
        eps = torch.finfo(y_pred.dtype).eps
        y_pred = torch.clamp(y_pred, eps, 1 - eps)

        return -torch.mean(
            y_true * torch.log(y_pred)
            + (1 - y_true) * torch.log(1 - y_pred)
        )

    def fit(self, X, y, n_epochs=100):
        X = torch.as_tensor(X, dtype=torch.float32)
        y = torch.as_tensor(y, dtype=torch.float32).reshape(-1, 1)

        n = X.shape[0]
        self.loss_history = []

        for epoch in range(n_epochs):
            # Stochastic gradient descent: shuffle observations
            randperm = torch.randperm(n)

            for ii in randperm:
                x_batch = X[ii:ii + 1]
                y_batch = y[ii:ii + 1]

                # Forward pass
                y_pred = self.forward(x_batch)

                # Output gradient: sigmoid + binary cross-entropy
                delta_out = y_pred - y_batch

                # Gradient for output layer
                if self.hidden_dim2 is not None:
                    dW_out = self.a_2.T @ delta_out
                else:
                    dW_out = self.a_1.T @ delta_out

                db_out = delta_out.sum(dim=0)

                # Backpropagate through optional second hidden layer
                if self.hidden_dim2 is not None:
                    if self.activation == "relu":
                        derivative_2 = (self.h_2 > 0).float()
                    else:
                        derivative_2 = 1 - self.a_2 ** 2

                    delta_2 = (delta_out @ self.W_out.T) * derivative_2

                    dW_2 = self.a_1.T @ delta_2
                    db_2 = delta_2.sum(dim=0)

                    if self.activation == "relu":
                        derivative_1 = (self.h_1 > 0).float()
                    else:
                        derivative_1 = 1 - self.a_1 ** 2

                    delta_1 = (delta_2 @ self.W_2.T) * derivative_1

                else:
                    if self.activation == "relu":
                        derivative_1 = (self.h_1 > 0).float()
                    else:
                        derivative_1 = 1 - self.a_1 ** 2

                    delta_1 = (delta_out @ self.W_out.T) * derivative_1

                dW_1 = x_batch.T @ delta_1
                db_1 = delta_1.sum(dim=0)

                # Manual gradient descent updates
                with torch.no_grad():
                    self.W_1 -= self.learning_rate * dW_1
                    self.b_1 -= self.learning_rate * db_1

                    self.W_out -= self.learning_rate * dW_out
                    self.b_out -= self.learning_rate * db_out

                    if self.hidden_dim2 is not None:
                        self.W_2 -= self.learning_rate * dW_2
                        self.b_2 -= self.learning_rate * db_2

            # Record loss at the end of each epoch
            with torch.no_grad():
                predictions = self.forward(X)
                epoch_loss = self.cost(predictions, y)
                self.loss_history.append(epoch_loss.item())

            if epoch % 10 == 0:
                print(
                    f"Epoch {epoch}, "
                    f"Cost: {self.loss_history[-1]:.4f}"
                )

        return self

    def predict(self, X):
        X = torch.as_tensor(X, dtype=torch.float32)

        with torch.no_grad():
            probabilities = self.forward(X)
            predicted = (probabilities >= 0.5).float()

        return predicted

In [5]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

# 1. Split into training and holdout sets (80/20)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# 2. Standardize features without data leakage
scaler = StandardScaler()

# Fit ONLY on the training features
X_train = scaler.fit_transform(X_train)

# Apply the same scaling to the holdout features
X_holdout = scaler.transform(X_holdout)

# 3. Convert features and labels into PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_holdout = torch.tensor(X_holdout, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_holdout = torch.tensor(y_holdout, dtype=torch.float32).reshape(-1, 1)

# Check the shapes
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_holdout:", X_holdout.shape)
print("y_holdout:", y_holdout.shape)

X_train: torch.Size([800, 2])
y_train: torch.Size([800, 1])
X_holdout: torch.Size([200, 2])
y_holdout: torch.Size([200, 1])


In [6]:

import pandas as pd
from sklearn.metrics import accuracy_score

# Fixed training settings
learning_rate = 0.01
epochs = 150

# Test different network sizes and activation functions
configs = []

# One hidden layer: vary the number of hidden units
for units in [8, 16, 32]:
    for activation in ["relu", "tanh"]:
        configs.append({
            "Architecture": f"1 hidden layer ({units})",
            "Hidden 1": units,
            "Hidden 2": None,
            "Activation": activation
        })

# Two hidden layers: use the same number of units in each layer
for units in [8, 16, 32]:
    for activation in ["relu", "tanh"]:
        configs.append({
            "Architecture": f"2 hidden layers ({units}, {units})",
            "Hidden 1": units,
            "Hidden 2": units,
            "Activation": activation
        })

results = []

for config in configs:
    # Create a fresh model for each experiment
    model = BinaryMLP(
        input_dim=X_train.shape[1],
        hidden_dim=config["Hidden 1"],
        hidden_dim2=config["Hidden 2"],
        learning_rate=learning_rate,
        activation=config["Activation"]
    )

    # Train the model
    model.fit(X_train, y_train, n_epochs=epochs)

    # Predict on both datasets
    train_pred = model.predict(X_train).detach().cpu().numpy().ravel()
    holdout_pred = model.predict(X_holdout).detach().cpu().numpy().ravel()

    # Convert true labels to NumPy arrays
    train_true = y_train.detach().cpu().numpy().ravel()
    holdout_true = y_holdout.detach().cpu().numpy().ravel()

    # Calculate accuracy
    train_accuracy = accuracy_score(train_true, train_pred)
    holdout_accuracy = accuracy_score(holdout_true, holdout_pred)

    results.append({
        "Architecture": config["Architecture"],
        "Activation": config["Activation"].title(),
        "Learning Rate": learning_rate,
        "Epochs": epochs,
        "Train Accuracy": train_accuracy,
        "Holdout Accuracy": holdout_accuracy
    })

# Display results, highest holdout accuracy first
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    "Holdout Accuracy", ascending=False
).reset_index(drop=True)

results_df.round(4)

Epoch 0, Cost: 0.5094
Epoch 10, Cost: 0.2883
Epoch 20, Cost: 0.2777
Epoch 30, Cost: 0.2314
Epoch 40, Cost: 0.1508
Epoch 50, Cost: 0.1041
Epoch 60, Cost: 0.0899
Epoch 70, Cost: 0.0846
Epoch 80, Cost: 0.0811
Epoch 90, Cost: 0.0802
Epoch 100, Cost: 0.0775
Epoch 110, Cost: 0.0811
Epoch 120, Cost: 0.0749
Epoch 130, Cost: 0.0730
Epoch 140, Cost: 0.0716
Epoch 0, Cost: 0.3398
Epoch 10, Cost: 0.2919
Epoch 20, Cost: 0.2906
Epoch 30, Cost: 0.2908
Epoch 40, Cost: 0.2894
Epoch 50, Cost: 0.2906
Epoch 60, Cost: 0.2890
Epoch 70, Cost: 0.2890
Epoch 80, Cost: 0.2901
Epoch 90, Cost: 0.2893
Epoch 100, Cost: 0.2889
Epoch 110, Cost: 0.2913
Epoch 120, Cost: 0.2886
Epoch 130, Cost: 0.2891
Epoch 140, Cost: 0.2926
Epoch 0, Cost: 0.3956
Epoch 10, Cost: 0.2857
Epoch 20, Cost: 0.2657
Epoch 30, Cost: 0.1962
Epoch 40, Cost: 0.1192
Epoch 50, Cost: 0.0936
Epoch 60, Cost: 0.0829
Epoch 70, Cost: 0.0776
Epoch 80, Cost: 0.0755
Epoch 90, Cost: 0.0728
Epoch 100, Cost: 0.0701
Epoch 110, Cost: 0.0689
Epoch 120, Cost: 0.0679
E

,Architecture,Activation,Learning Rate,Epochs,Train Accuracy,Holdout Accuracy
0,1 hidden layer (16),Tanh,0.01,150,0.9688,0.995
1,"2 hidden layers (16, 16)",Tanh,0.01,150,0.9612,0.995
2,1 hidden layer (32),Tanh,0.01,150,0.9575,0.990
3,1 hidden layer (8),Relu,0.01,150,0.9725,0.985
4,"2 hidden layers (16, 16)",Relu,0.01,150,0.9762,0.985
5,1 hidden layer (16),Relu,0.01,150,0.9700,0.985
6,1 hidden layer (32),Relu,0.01,150,0.9762,0.985
7,"2 hidden layers (32, 32)",Relu,0.01,150,0.9788,0.980
8,"2 hidden layers (8, 8)",Relu,0.01,150,0.9675,0.975
9,1 hidden layer (8),Tanh,0.01,150,0.8700,0.865


Across the board, we can see that increasing the amount of hidden units benefitted performance. However, models with smaller hidden units still performed well. This means that a relatively small network can already learn patterns in this dataset.

Similarly, two layers did not consistently outperform one layer. Some of these models actually performed worse with two layers. For example, the Tanh model with 2 hidden layers of 32 units underperformed the Tanh model with 1 hidden layer of 32 units.

The performance of both was dependent on the architecture. ReLU consistently performed well across all architectures. On the other hand Tanh had two comparatively poor performing models, but it also had good ones. The two worst were the ones with two hidden layers and eight and 32 hidden units. The results suggest that the best activation function is based on architecture.

## 2. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.  

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:  
   - Label = 0 → [1, 0, 0]  
   - Label = 1 → [0, 1, 0]  
   - Label = 2 → [0, 0, 1]  
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.  

 - The **predicted class** corresponds to the unit with the highest probability.  
   Example:  
   - `[0.1, 0.3, 0.6] → class 2`  
   - `[0.6, 0.2, 0.2] → class 0`  

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:  

   $$\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [7]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_classes=3,
                           n_clusters_per_class=1,
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [8]:
# USE AS MANY CELLS AS NEEDED

## 3: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [9]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

In [10]:
# USE AS MANY CELLS AS NEEDED

## 4. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

YOUR TEXT HERE

## 5. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE